In [31]:
from pathlib import Path
import json
import hcl2
import difflib
import yaml
import webbrowser
import networkx as nx
from pyvis.network import Network
from zss import Node, simple_distance
from itertools import combinations
from clone_detection_parallel import detect_clones_zss as detect_clones_zss_parallel


In [32]:
# This will hold ASTs to avoid re-parsing
parsed_asts = {}

def find_iac_files(root):
    exts = ('.tf',)
    return [p for p in Path(root).rglob('*') if p.is_file() and p.suffix in exts]

def parse_file(path):
    global parsed_asts
    if path in parsed_asts:
        return parsed_asts[path]
    
    try:
        with open(path, 'r', encoding='utf-8') as f:
            if path.suffix == '.tf':
                data = hcl2.load(f)
            elif path.suffix in ('.yaml', '.yml'):
                data = yaml.safe_load(f)
            elif path.suffix == '.json':
                data = json.load(f)
        
        parsed_asts[path] = data
        return data
    except Exception as e:
        print(f"Failed to parse {path}: {e}")
        return None

def to_zss_tree(node, label='root'):
    """
    Recursively convert a dictionary/list-based AST into a zss.Node tree.
    """
    if isinstance(node, dict):
        zss_node = Node(label)
        for k, v in sorted(node.items()): # Sort keys for consistency
            zss_node.addkid(to_zss_tree(v, label=k))
        return zss_node
    elif isinstance(node, list):
        zss_node = Node(label)
        for item in node:
            # Use a generic label for list items
            zss_node.addkid(to_zss_tree(item, label='item'))
        return zss_node
    else:
        # For literals, you can use their type or a constant value
        return Node(str(type(node).__name__))

def detect_clones_zss(files, threshold=5):
    """
    Detects clones using tree edit distance with zss.
    Returns a list of clone pairs and their distance.
    """
    asts = {}
    for path in files:
        ast_tree = parse_file(path)
        if ast_tree:
            asts[path] = to_zss_tree(ast_tree)

    clone_pairs = []
    # Compare every file with every other file
    for path1, path2 in combinations(asts.keys(), 2):
        distance = simple_distance(asts[path1], asts[path2])

        if distance <= threshold:
            print(f"Found potential clone pair: ({path1.name}, {path2.name}) with distance {distance}")
            clone_pairs.append((path1, path2, distance))

    return clone_pairs

def report_zss(clone_groups):
    if not clone_groups:
        print("No clones found.")
        return
    
    for i, group in enumerate(clone_groups):
        print(f"\nClone group {i+1}: {len(group)} files")
        for p in sorted(list(group)): # Sort for consistent output
            print(f"  - {p}")


def visualize_clones_diff(clone_groups):
    """
    Generates a text-based diff for each pair of files in a clone group.
    """
    if not clone_groups:
        print("No clones to visualize.")
        return

    print(f"\n--- Visualizing {len(clone_groups)} Clone Groups ---")

    for i, group in enumerate(clone_groups):
        # Iterate over every pair of files in the clone group
        for path1, path2 in combinations(sorted(list(group)), 2):
            print(f"\n{'='*80}")
            print(f"Clone Group {i+1}: DIFF between '{path1.name}' and '{path2.name}'")
            print(f"{'='*80}")

            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()

                # Generate and print the diff
                diff = difflib.unified_diff(
                    file1_lines,
                    file2_lines,
                    fromfile=str(path1),
                    tofile=str(path2),
                )
                for line in diff:
                    print(line, end='')

            except Exception as e:
                print(f"Could not generate diff for {path1.name} and {path2.name}: {e}")

def _find_ast_diff(d1, d2, path=""):
    """Helper to find differing values in two ASTs (dicts)."""
    diffs = set()
    # Check keys in d1 against d2
    for k in d1:
        p = f"{path}.{k}" if path else k
        if k not in d2:
            diffs.add(f"Key '{p}' only in first file.")
            continue
        
        v1, v2 = d1[k], d2[k]
        if isinstance(v1, dict) and isinstance(v2, dict):
            diffs.update(_find_ast_diff(v1, v2, path=p))
        elif isinstance(v1, list) and isinstance(v2, list):
            if len(v1) != len(v2):
                diffs.add(f"List length differs at '{p}' ({len(v1)} vs {len(v2)})")
            else:
                # Iterate through list items
                for i, (item1, item2) in enumerate(zip(v1, v2)):
                    item_path = f"{p}[{i}]"
                    if isinstance(item1, dict) and isinstance(item2, dict):
                        diffs.update(_find_ast_diff(item1, item2, path=item_path))
                    elif item1 != item2:
                        diffs.add(f"Value differs at '{item_path}': ('{item1}' vs '{item2}')")
        elif v1 != v2:
            diffs.add(f"Value differs at '{p}': ('{v1}' vs '{v2}')")

    # Check for keys in d2 that are not in d1
    for k in d2:
        p = f"{path}.{k}" if path else k
        if k not in d1:
            diffs.add(f"Key '{p}' only in second file.")
            
    return diffs


def _generate_module_tf(base_ast, differences):
    """Generates the HCL for a new Terraform module."""
    variables = {}
    # Naively create variable names from the diff path
    for diff in differences:
        # Example diff: "Value differs at '.resource.aws_instance.web.instance_type': ('t2.micro' vs 't2.large')"
        try:
            path_part = diff.split("'")[1]
            # A simple heuristic for variable names
            var_name = path_part.split('.')[-1]
            variables[var_name] = {'path': path_part}
        except IndexError:
            continue

    # --- Generate variables.tf content ---
    var_tf_lines = ['# variables.tf for the new module\n']
    for name in sorted(variables.keys()):
        var_tf_lines.append(f'variable "{name}" {{')
        var_tf_lines.append('  description = "Autogenerated variable"')
        var_tf_lines.append('}\n')

    # --- Generate main.tf content (very simplified) ---
    # This is a complex task. For this example, we'll just show the concept
    # by replacing values in a string representation of the original resource.
    # A real implementation would need a robust HCL generator.
    main_tf_lines = ['# main.tf for the new module\n']
    
    # Handle that 'resource' can be a list of dicts or a single dict
    resource_block = base_ast.get('resource')
    if isinstance(resource_block, list):
        resource_block = resource_block[0] # Use the first resource block as a template
    
    # Let's find the first resource block to use as a template
    resource_key = next((k for k in resource_block), None) if resource_block else None
    if resource_key:
        resource_name = next(iter(resource_block[resource_key]), None)
        if resource_name:
            # Crude representation of the resource
            main_tf_lines.append(f'resource "{resource_key}" "{resource_name}" {{')
            
            # The resource block can be a list with one dict or just the dict
            resource_attributes = resource_block[resource_key][resource_name]
            if isinstance(resource_attributes, list):
                resource_attributes = resource_attributes[0]

            for key, value in resource_attributes.items():
                 # Check if this attribute is a variable
                is_variable = False
                for var_name, var_info in variables.items():
                    if var_info['path'].endswith(f'.{key}'):
                        main_tf_lines.append(f'  {key} = var.{var_name}')
                        is_variable = True
                        break
                if not is_variable:
                    main_tf_lines.append(f'  {key} = "{value}"') # Note: simplistic quoting
            main_tf_lines.append('}')

    return '\n'.join(var_tf_lines), '\n'.join(main_tf_lines)


def _generate_module_call(module_name, differences, original_values):
    """Generates the HCL for calling the new module."""
    call_lines = [f'module "{module_name}" {{', '  source = "./modules/{module_name}"\n']
    
    variables = {}
    for diff in differences:
        try:
            path_part = diff.split("'")[1]
            var_name = path_part.split('.')[-1]
            # Extract the first value as an example
            value = diff.split("'")[3]
            variables[var_name] = value
        except IndexError:
            continue

    for name, value in sorted(variables.items()):
        call_lines.append(f'  {name} = "{value}"')

    call_lines.append('}')
    return '\n'.join(call_lines)


In [33]:

def suggest_refactorings(clone_groups):
    """Analyzes clone groups and suggests refactoring into modules."""
    if not clone_groups:
        return

    print(f"\n{'='*80}")
    print("Refactoring Suggestions")
    print(f"{'='*80}")

    for i, group in enumerate(clone_groups):
        if len(group) < 2:
            continue

        print(f"\n--- Clone Group {i+1} ---")
        print("Suggestion: These files are strong candidates for extraction into a reusable Terraform module.")

        # Analyze differences between the first two files in the group as an example
        paths = sorted(list(group))
        path1, path2 = paths[0], paths[1]
        
        ast1 = parse_file(path1)
        ast2 = parse_file(path2)

        if ast1 and ast2:
            differences = _find_ast_diff(ast1, ast2)
            if differences:
                print("\nPotential variables for the module (based on differences):")
                for diff in sorted(list(differences))[:5]: # Show top 5 differences
                    print(f"  - {diff}")
                
                print("\n--- Proposed 'variables.tf' for new module ---")
                variables_tf, main_tf = _generate_module_tf(ast1, differences)
                print(variables_tf)

                print("\n--- Proposed 'main.tf' for new module ---")
                print(main_tf)

                print("\n--- Proposed module call (replaces original code) ---")
                module_call = _generate_module_call(f"refactored_group_{i+1}", differences, ast1)
                print(module_call)

            else:
                print("Files appear structurally identical but may have formatting differences.")


In [34]:


def visualize_clones_html(clone_groups, output_filename="clone_diff_report.html"):
    """
    Generates an HTML file with side-by-side diffs for clone pairs.
    """
    if not clone_groups:
        print("No clones to visualize.")
        return

    html_parts = [
        '<html><head><title>Clone Diff Report</title>',
        '<style>body { font-family: sans-serif; } table { border-collapse: collapse; }',
        'td { padding: 2px; } .diff_header { background-color: #e0e0e0; }',
        'h1, h2 { padding: 10px; background-color: #f0f0f0; border-radius: 5px; }',
        '</style></head><body><h1>Clone Detection Report</h1>'
    ]

    # Use HtmlDiff for a side-by-side comparison
    html_diff = difflib.HtmlDiff(tabsize=4, wrapcolumn=50)

    for i, group in enumerate(clone_groups):
        html_parts.append(f"<h2>Clone Group {i+1}</h2>")
        for path1, path2 in combinations(sorted(list(group)), 2):
            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()

                # Generate the HTML table for the diff
                diff_table = html_diff.make_table(
                    file1_lines,
                    file2_lines,
                    fromdesc=str(path1.name),
                    todesc=str(path2.name)
                )
                html_parts.append(diff_table)

            except Exception as e:
                html_parts.append(f"<p>Could not generate diff for {path1.name} and {path2.name}: {e}</p>")

    html_parts.append('</body></html>')

    # Write the final HTML file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_parts))

    print(f"HTML report generated: {output_filename}")
    # Open the report in the default web browser
    webbrowser.open(f"file://{Path(output_filename).resolve()}")


def visualize_clone_graph(clone_pairs, output_filename="clone_graph.html"):
    """
    Generates an interactive HTML graph of clone relationships.
    Writes the generated HTML using UTF-8 to avoid encoding errors on Windows.
    """
    if not clone_pairs:
        print("No clone pairs to visualize in a graph.")
        return

    net = Network(notebook=True, directed=False, cdn_resources='in_line')
    
    # Use NetworkX to handle graph properties like connected components (groups)
    G = nx.Graph()
    for path1, path2, distance in clone_pairs:
        G.add_edge(str(path1.name), str(path2.name), weight=distance, title=f"Distance: {distance}")

    # Assign a group ID to each component for coloring
    for i, component in enumerate(nx.connected_components(G)):
        for node in component:
            if node in G:
                G.nodes[node]['group'] = i

    net.from_nx(G)
    net.show_buttons(filter_=['physics'])

    # Generate the HTML and write it explicitly with UTF-8 encoding to avoid
    # UnicodeEncodeError on platforms that use a limited default encoding.
    try:
        html = net.generate_html()
        with open(output_filename, 'w', encoding='utf-8') as f:
            f.write(html)
        print(f"Interactive graph saved to {output_filename}")
        webbrowser.open(f"file://{Path(output_filename).resolve()}")
    except Exception as e:
        print(f"Failed to generate or save interactive graph: {e}")


In [35]:
def generate_comprehensive_report(clone_pairs, clone_groups, output_filename="clone_report.html"):
    """
    Generates a comprehensive HTML report with all visualizations:
    - Overview statistics
    - Interactive clone graph (embedded)
    - Side-by-side diffs
    - Refactoring suggestions
    """
    if not clone_pairs and not clone_groups:
        print("No clones to report.")
        return

    html_parts = [
        '<!DOCTYPE html><html><head><meta charset="UTF-8">',
        '<title>Clone Detection Report</title>',
        '<style>',
        '* { margin: 0; padding: 0; box-sizing: border-box; }',
        'body { font-family: "Segoe UI", Tahoma, Geneva, Verdana, sans-serif; background: #f5f5f5; color: #333; }',
        'header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 2rem; box-shadow: 0 2px 10px rgba(0,0,0,0.1); }',
        'header h1 { font-size: 2.5rem; margin-bottom: 0.5rem; }',
        'header p { font-size: 1.1rem; opacity: 0.9; }',
        '.container { max-width: 1400px; margin: 0 auto; padding: 2rem; }',
        '.stats { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 1rem; margin-bottom: 2rem; }',
        '.stat-card { background: white; padding: 1.5rem; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); text-align: center; }',
        '.stat-card h3 { color: #667eea; font-size: 2rem; margin-bottom: 0.5rem; }',
        '.stat-card p { color: #666; font-size: 0.9rem; }',
        'nav { background: white; padding: 1rem; border-radius: 8px; margin-bottom: 2rem; box-shadow: 0 2px 8px rgba(0,0,0,0.1); position: sticky; top: 20px; z-index: 100; }',
        'nav a { color: #667eea; text-decoration: none; padding: 0.5rem 1rem; margin: 0 0.25rem; border-radius: 4px; transition: background 0.3s; display: inline-block; }',
        'nav a:hover { background: #f0f0f0; }',
        '.section { background: white; padding: 2rem; margin-bottom: 2rem; border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.1); }',
        '.section h2 { color: #667eea; margin-bottom: 1.5rem; padding-bottom: 0.5rem; border-bottom: 2px solid #667eea; }',
        '.clone-group { background: #f9f9f9; padding: 1.5rem; margin-bottom: 1.5rem; border-radius: 8px; border-left: 4px solid #667eea; }',
        '.clone-group h3 { color: #764ba2; margin-bottom: 1rem; }',
        '.file-list { background: white; padding: 1rem; border-radius: 4px; margin-bottom: 1rem; }',
        '.file-list li { padding: 0.5rem; border-bottom: 1px solid #eee; list-style-position: inside; }',
        '.file-list li:last-child { border-bottom: none; }',
        'table.diff { font-family: "Courier New", monospace; font-size: 0.85rem; width: 100%; border: 1px solid #ddd; margin: 1rem 0; }',
        'table.diff td { padding: 0.25rem 0.5rem; white-space: pre-wrap; word-wrap: break-word; }',
        '.diff_header { background-color: #667eea !important; color: white !important; font-weight: bold; }',
        '.diff_next { background-color: #e0e0e0; }',
        '.diff_add { background-color: #d4edda; }',
        '.diff_chg { background-color: #fff3cd; }',
        '.diff_sub { background-color: #f8d7da; }',
        'pre { background: #f4f4f4; padding: 1rem; border-radius: 4px; overflow-x: auto; font-size: 0.9rem; }',
        'code { background: #f4f4f4; padding: 0.2rem 0.4rem; border-radius: 3px; font-family: "Courier New", monospace; }',
        '.refactoring { background: #e7f3ff; padding: 1rem; border-radius: 4px; margin: 1rem 0; border-left: 4px solid #2196F3; }',
        '.graph-container { width: 100%; height: 600px; border: 1px solid #ddd; border-radius: 4px; margin: 1rem 0; }',
        'footer { text-align: center; padding: 2rem; color: #666; font-size: 0.9rem; }',
        '</style>',
        '</head><body>',
        '<header>',
        '<div class="container">',
        '<h1> Clone Detection Report</h1>',
        '</div>',
        '</header>',
        '<div class="container">'
    ]

    # Statistics section
    total_files = len(set([p for pair in clone_pairs for p in [pair[0], pair[1]]]))
    total_groups = len(clone_groups)
    total_pairs = len(clone_pairs)
    
    html_parts.extend([
        '<div class="stats">',
        '<div class="stat-card"><h3>{}</h3><p>Clone Groups</p></div>'.format(total_groups),
        '<div class="stat-card"><h3>{}</h3><p>Clone Pairs</p></div>'.format(total_pairs),
        '<div class="stat-card"><h3>{}</h3><p>Files Involved</p></div>'.format(total_files),
        '</div>'
    ])

    # Navigation
    html_parts.extend([
        '<nav>',
        '<a href="#graph"> Graph View</a>',
        '<a href="#diffs"> Code Diffs</a>',
        '<a href="#refactoring"> Refactoring</a>',
        '</nav>'
    ])

    # Clone Graph Section
    html_parts.extend([
        '<div class="section" id="graph">',
        '<h2> Clone Relationship Graph</h2>',
        '<div class="graph-container" id="mynetwork"></div>',
        '</div>'
    ])

    # Generate the Pyvis network for embedding
    if clone_pairs:
        net = Network(height="600px", width="100%", directed=False, cdn_resources='in_line')
        G = nx.Graph()
        for path1, path2, distance in clone_pairs:
            G.add_edge(str(path1.name), str(path2.name), weight=distance, title=f"Distance: {distance}")
        
        for i, component in enumerate(nx.connected_components(G)):
            for node in component:
                if node in G:
                    G.nodes[node]['group'] = i
        
        net.from_nx(G)
        
        # Extract the network HTML and inject it
        try:
            graph_html = net.generate_html()
            # Extract just the script and style parts
            import re
            script_match = re.search(r'<script type="text/javascript">(.*?)</script>', graph_html, re.DOTALL)
            if script_match:
                html_parts.append(f'<script type="text/javascript">{script_match.group(1)}</script>')
        except Exception as e:
            html_parts.append(f'<p>Could not embed graph: {e}</p>')

    # Code Diffs Section
    html_parts.extend([
        '<div class="section" id="diffs">',
        '<h2> Side-by-Side Code Comparisons</h2>'
    ])

    html_diff = difflib.HtmlDiff(tabsize=4, wrapcolumn=50)
    for i, group in enumerate(clone_groups):
        html_parts.append(f'<div class="clone-group">')
        html_parts.append(f'<h3>Clone Group {i+1}</h3>')
        html_parts.append('<ul class="file-list">')
        for path in sorted(list(group)):
            html_parts.append(f'<li>{path}</li>')
        html_parts.append('</ul>')
        
        for path1, path2 in combinations(sorted(list(group)), 2):
            try:
                with open(path1, 'r', encoding='utf-8') as f1, open(path2, 'r', encoding='utf-8') as f2:
                    file1_lines = f1.readlines()
                    file2_lines = f2.readlines()
                
                diff_table = html_diff.make_table(
                    file1_lines,
                    file2_lines,
                    fromdesc=str(path1.name),
                    todesc=str(path2.name),
                    context=True,
                    numlines=2
                )
                html_parts.append(diff_table)
            except Exception as e:
                html_parts.append(f'<p> Could not generate diff: {e}</p>')
        
        html_parts.append('</div>')
    
    html_parts.append('</div>')

    # Refactoring Suggestions Section
    html_parts.extend([
        '<div class="section" id="refactoring">',
        '<h2> Refactoring Suggestions</h2>'
    ])

    for i, group in enumerate(clone_groups):
        if len(group) < 2:
            continue
        
        html_parts.append(f'<div class="clone-group">')
        html_parts.append(f'<h3>Clone Group {i+1} - Module Extraction</h3>')
        html_parts.append('<div class="refactoring">')
        html_parts.append('<p><strong> Suggestion:</strong> Extract these files into a reusable Terraform module.</p>')
        
        paths = sorted(list(group))
        path1, path2 = paths[0], paths[1]
        ast1 = parse_file(path1)
        ast2 = parse_file(path2)

        if ast1 and ast2:
            differences = _find_ast_diff(ast1, ast2)
            if differences:
                html_parts.append('<p><strong>Potential module variables:</strong></p>')
                html_parts.append('<ul>')
                for diff in sorted(list(differences))[:5]:
                    html_parts.append(f'<li><code>{diff}</code></li>')
                html_parts.append('</ul>')
                
                variables_tf, main_tf = _generate_module_tf(ast1, differences)
                module_call = _generate_module_call(f"refactored_group_{i+1}", differences, ast1)
                
                html_parts.append('<h4>Proposed variables.tf:</h4>')
                html_parts.append(f'<pre><code>{variables_tf}</code></pre>')
                
                html_parts.append('<h4>Proposed main.tf:</h4>')
                html_parts.append(f'<pre><code>{main_tf}</code></pre>')
                
                html_parts.append('<h4>Module call (replaces original):</h4>')
                html_parts.append(f'<pre><code>{module_call}</code></pre>')
            else:
                html_parts.append('<p>Files are structurally identical.</p>')
        
        html_parts.append('</div></div>')
    
    html_parts.append('</div>')

    # Footer
    html_parts.extend([
        '</div>',
        '<footer>Generated by Clone Detection Tool</footer>',
        '</body></html>'
    ])

    # Write the file
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(html_parts))

    print(f"Comprehensive report generated: {output_filename}")
    webbrowser.open(f"file://{Path(output_filename).resolve()}")


In [36]:
root_dir = 'C:/Users/Falco/Documents/Università/EQS/terraform-examples/'  # Change as needed
iac_files = find_iac_files(root_dir)


In [37]:
# You can adjust the threshold. A lower value means stricter cloning.
# threshold=0 would find only structurally identical files.
# clone_pairs_with_distance = detect_clones_zss(iac_files, threshold=2)
clone_pairs_with_distance = detect_clones_zss_parallel(iac_files, threshold=2)


Parsing files and building ASTs...
Failed to parse C:\Users\Falco\Documents\Università\EQS\terraform-examples\google_cloud\camunda\build.tf: Unexpected token Token('STRING_CHARS', '\n      sha1(\n        ') at line 42, column 91.
Expected one of: 
	* MINUS
	* FOR_EACH
	* NAME
	* FOR
	* IN
	* /<<(?P<heredoc>[a-zA-Z][a-zA-Z0-9._-]+)\n?(?:.|\n)*?\n\s*(?P=heredoc)\n/
	* LPAR
	* LSQB
	* DBLQUOTE
	* DECIMAL
	* BANG
	* NEGATIVE_DECIMAL
	* /<<-(?P<heredoc_trim>[a-zA-Z][a-zA-Z0-9._-]+)\n?(?:.|\n)*?\n\s*(?P=heredoc_trim)\n/
	* IF
	* LBRACE
Previous tokens: [Token('__ANON_0', '${')]

Failed to parse C:\Users\Falco\Documents\Università\EQS\terraform-examples\google_cloud\camunda-secure\build.tf: Unexpected token Token('STRING_CHARS', '\n      sha1(\n        ') at line 42, column 89.
Expected one of: 
	* MINUS
	* FOR_EACH
	* NAME
	* FOR
	* IN
	* /<<(?P<heredoc>[a-zA-Z][a-zA-Z0-9._-]+)\n?(?:.|\n)*?\n\s*(?P=heredoc)\n/
	* LPAR
	* LSQB
	* DBLQUOTE
	* DECIMAL
	* BANG
	* NEGATIVE_DECIMAL
	* /<<-(?P<here

In [38]:

# We need to reconstruct the clone groups for the other functions
clone_groups_from_pairs = list(nx.connected_components(nx.Graph([(p1, p2) for p1, p2, _ in clone_pairs_with_distance])))

report_zss(clone_groups_from_pairs)



Clone group 1: 2 files
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_lambda_api\certificate.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_reverse_proxy\certificate.tf

Clone group 2: 3 files
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_lambda_api\data.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_mailgun_domain\data.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_reverse_proxy\data.tf

Clone group 3: 2 files
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_lambda_api\main.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_lambda_cronjob\main.tf

Clone group 4: 3 files
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_lambda_api\outputs.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-examples\aws\aws_mailgun_domain\outputs.tf
  - C:\Users\Falco\Documents\Università\EQS\terraform-ex

In [39]:
#visualize_clones_diff(clone_groups_from_pairs)


In [40]:
#visualize_clones_html(clone_groups_from_pairs)


In [41]:
#visualize_clone_graph(clone_pairs_with_distance)

In [42]:
#suggest_refactorings(clone_groups_from_pairs)

In [43]:
# Generate the comprehensive report with all visualizations
generate_comprehensive_report(clone_pairs_with_distance, clone_groups_from_pairs)

Comprehensive report generated: clone_report.html
